# 05. Workflow Orchestration - Microsoft Agent Framework Edition

## 개요
Microsoft Agent Framework의 핵심 기능인 Graph-based Workflow를 사용하여
복잡한 멀티 에이전트 시스템을 구조화하고 오케스트레이션합니다.

### 학습 목표
- Workflow 기본 개념 이해
- Graph-based 오케스트레이션
- 조건부 라우팅
- 병렬 처리
- 체크포인팅 및 상태 관리

## 1. 환경 설정

In [18]:
import os
import asyncio
from dotenv import load_dotenv
from agent_framework import ChatAgent, Workflow, WorkflowBuilder, WorkflowContext
from agent_framework.azure import AzureOpenAIChatClient
from typing import Dict, Any

# 환경변수 로드
load_dotenv()

# Azure OpenAI Chat Client 생성
chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


## 2. 기본 워크플로우 생성

간단한 선형 워크플로우를 만들어봅니다.

In [19]:
# 에이전트 생성
preprocessor = ChatAgent(
    chat_client=chat_client,
    instructions="Extract key information and structure the input data.",
    name="Preprocessor"
)

analyzer = ChatAgent(
    chat_client=chat_client,
    instructions="Analyze the preprocessed data and identify patterns.",
    name="Analyzer"
)

reporter = ChatAgent(
    chat_client=chat_client,
    instructions="Generate a comprehensive report based on the analysis.",
    name="Reporter"
)

# 워크플로우 빌더로 구성 (lazy initialization 사용)
builder = WorkflowBuilder()
builder.register_agent(lambda: preprocessor, name="preprocessor")
builder.register_agent(lambda: analyzer, name="analyzer")
builder.register_agent(lambda: reporter, name="reporter")

# 에이전트들 간 연결
builder.add_edge("preprocessor", "analyzer")
builder.add_edge("analyzer", "reporter")

# 시작점 설정
builder.set_start_executor("preprocessor")

# 워크플로우 빌드
workflow = builder.build()

print("✅ 기본 워크플로우 생성 완료")
print("   Flow: Preprocessor → Analyzer → Reporter")

✅ 기본 워크플로우 생성 완료
   Flow: Preprocessor → Analyzer → Reporter


In [20]:
# 워크플로우 실행
async def run_basic_workflow():
    user_input = "Analyze the trends in cloud computing adoption in 2025"
    
    print(f"Input: {user_input}\n")
    print("Workflow execution:")
    
    # 워크플로우 실행 및 결과 수집
    results = []
    async for event in workflow.run_stream(user_input):
        print(f"Event: {event}")
        results.append(event)
    
    print("\n✅ Workflow completed successfully")

await run_basic_workflow()

Input: Analyze the trends in cloud computing adoption in 2025

Workflow execution:
Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)
Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)
Event: ExecutorInvokedEvent(executor_id=Preprocessor, data=Analyze the trends in cloud computing adoption in 2025)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages=)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages=Key)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages= Trends)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages= in)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages= Cloud)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages= Computing)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages= Adoption)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, messages= in)
Event: AgentRunUpdateEvent(executor_id=Preprocessor, message

## 3. 조건부 라우팅

상태에 따라 다른 경로로 실행을 분기합니다.

In [21]:
# 분류 에이전트
classifier = ChatAgent(
    chat_client=chat_client,
    instructions="""
    Classify the input into categories: TECHNICAL, BUSINESS, or GENERAL.
    Respond with only one word: TECHNICAL, BUSINESS, or GENERAL.
    """,
    name="Classifier"
)

# 전문 분석 에이전트들
technical_analyst = ChatAgent(
    chat_client=chat_client,
    instructions="Provide technical analysis with focus on architecture and implementation.",
    name="TechnicalAnalyst"
)

business_analyst = ChatAgent(
    chat_client=chat_client,
    instructions="Provide business analysis with focus on ROI and market impact.",
    name="BusinessAnalyst"
)

general_analyst = ChatAgent(
    chat_client=chat_client,
    instructions="Provide general overview and high-level insights.",
    name="GeneralAnalyst"
)

# 라우팅 함수
def route_by_category(message: str) -> str:
    """카테고리에 따라 다음 에이전트 결정"""
    # 메시지를 분류하기 위해 키워드 확인
    message_upper = message.upper()
    if any(word in message_upper for word in ["TECHNICAL", "ARCHITECTURE", "CODE", "IMPLEMENT"]):
        return "technical"
    elif any(word in message_upper for word in ["BUSINESS", "ROI", "MARKET", "COST"]):
        return "business"
    else:
        return "general"

# 조건부 워크플로우 구성
conditional_builder = WorkflowBuilder()
conditional_builder.register_agent(lambda: classifier, name="classifier")
conditional_builder.register_agent(lambda: technical_analyst, name="technical")
conditional_builder.register_agent(lambda: business_analyst, name="business")
conditional_builder.register_agent(lambda: general_analyst, name="general")

# 분류 에이전트에서 출발하여 라우팅
conditional_builder.add_edge("classifier", "technical", 
                            condition=lambda msg: route_by_category(str(msg)) == "technical")
conditional_builder.add_edge("classifier", "business", 
                            condition=lambda msg: route_by_category(str(msg)) == "business")
conditional_builder.add_edge("classifier", "general", 
                            condition=lambda msg: route_by_category(str(msg)) == "general")

conditional_builder.set_start_executor("classifier")

conditional_workflow = conditional_builder.build()

print("✅ 조건부 워크플로우 생성 완료")

✅ 조건부 워크플로우 생성 완료


In [22]:
# 다양한 입력으로 테스트
async def test_conditional_routing():
    test_inputs = [
        "How should we implement microservices architecture?",
        "What is the ROI of implementing AI in our business?",
        "Tell me about climate change"
    ]
    
    for user_input in test_inputs:
        print(f"\n{'='*60}")
        print(f"Input: {user_input}")
        print(f"{'='*60}")
        
        # Determine routing
        routing = route_by_category(user_input)
        print(f"Routed to: {routing.upper()}")
        
        # Execute workflow
        async for event in conditional_workflow.run_stream(user_input):
            print(f"Event: {event}\n")

await test_conditional_routing()


Input: How should we implement microservices architecture?
Routed to: TECHNICAL
Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)

Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)

Event: ExecutorInvokedEvent(executor_id=Classifier, data=How should we implement microservices architecture?)

Event: AgentRunUpdateEvent(executor_id=Classifier, messages=)

Event: AgentRunUpdateEvent(executor_id=Classifier, messages=TECH)

Event: AgentRunUpdateEvent(executor_id=Classifier, messages=N)

Event: AgentRunUpdateEvent(executor_id=Classifier, messages=ICAL)

Event: AgentRunUpdateEvent(executor_id=Classifier, messages=)

Event: AgentRunUpdateEvent(executor_id=Classifier, messages=)

Event: ExecutorCompletedEvent(executor_id=Classifier, data=[AgentExecutorResponse(executor_id='Classifier', agent_response=<agent_framework._types.AgentResponse object at 0x7041c1e9bbf0>, full_conversation=[<agent_framework._types

## 4. 병렬 처리 워크플로우

여러 분석을 동시에 수행하고 결과를 통합합니다.

In [23]:
# 병렬 분석 에이전트들
technical_analyst_parallel = ChatAgent(
    chat_client=chat_client,
    instructions="Provide technical analysis with focus on architecture and implementation.",
    name="TechnicalAnalystParallel"
)

business_analyst_parallel = ChatAgent(
    chat_client=chat_client,
    instructions="Provide business analysis with focus on ROI and market impact.",
    name="BusinessAnalystParallel"
)

market_analyst = ChatAgent(
    chat_client=chat_client,
    instructions="Analyze market trends and competitive landscape.",
    name="MarketAnalyst"
)

# 병렬 워크플로우 구성
# 간단한 구조로 변경: 3개 에이전트를 독립적으로 실행
parallel_builder = WorkflowBuilder()
parallel_builder.register_agent(lambda: technical_analyst_parallel, name="technical", output_response=True)
parallel_builder.register_agent(lambda: business_analyst_parallel, name="business", output_response=True)
parallel_builder.register_agent(lambda: market_analyst, name="market", output_response=True)

# 세 에이전트가 동시에 같은 입력을 받을 수 있도록 설정
parallel_builder.add_fan_out_edges("technical", ["business", "market"])

parallel_builder.set_start_executor("technical")

parallel_workflow = parallel_builder.build()

print("✅ 병렬 처리 워크플로우 생성 완료")

✅ 병렬 처리 워크플로우 생성 완료


In [24]:
# 병렬 워크플로우 실행
async def run_parallel_workflow():
    topic = "Implementing AI agents in enterprise customer service"
    
    print(f"Topic: {topic}\n")
    print("📊 Multi-Perspective Analysis (Parallel Execution):\n")
    
    # 워크플로우 실행 및 결과 수집
    async for event in parallel_workflow.run_stream(topic):
        print(f"Event: {event}\n")
    
    print("\n✅ Parallel workflow completed successfully")

await run_parallel_workflow()

Topic: Implementing AI agents in enterprise customer service

📊 Multi-Perspective Analysis (Parallel Execution):

Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)

Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)

Event: ExecutorInvokedEvent(executor_id=TechnicalAnalystParallel, data=Implementing AI agents in enterprise customer service)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages=)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages=Implement)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages=ing)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages= AI)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages= agents)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages= in)

Event: AgentRunUpdateEvent(executor_id=TechnicalAnalystParallel, messages= enterp

## 5. 체크포인팅과 재시작

장기 실행 워크플로우의 상태를 저장하고 복원합니다.

In [25]:
import json
from pathlib import Path
from agent_framework import FileCheckpointStorage

# 체크포인팅 설정
checkpoint_dir = Path("./checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

# 장기 실행 에이전트들 (시뮬레이션)
data_collector = ChatAgent(
    chat_client=chat_client,
    instructions="Collect and structure information from multiple sources.",
    name="DataCollector"
)

data_processor = ChatAgent(
    chat_client=chat_client,
    instructions="Process and clean the collected data.",
    name="DataProcessor"
)

report_generator = ChatAgent(
    chat_client=chat_client,
    instructions="Generate a comprehensive report from processed data.",
    name="ReportGenerator"
)

# 체크포인팅이 적용된 워크플로우 구성
checkpoint_builder = WorkflowBuilder()
checkpoint_builder.register_agent(lambda: data_collector, name="collect")
checkpoint_builder.register_agent(lambda: data_processor, name="process")
checkpoint_builder.register_agent(lambda: report_generator, name="report")

# 순차적 연결
checkpoint_builder.add_edge("collect", "process")
checkpoint_builder.add_edge("process", "report")

checkpoint_builder.set_start_executor("collect")

# 체크포인팅 활성화
checkpoint_storage = FileCheckpointStorage(str(checkpoint_dir))
checkpoint_builder.with_checkpointing(checkpoint_storage)

checkpoint_workflow = checkpoint_builder.build()

print("✅ 체크포인팅 시스템 준비 완료")

✅ 체크포인팅 시스템 준비 완료


In [26]:
# 체크포인팅 워크플로우 실행
async def demo_checkpointing():
    user_input = "Analyze cloud computing adoption trends and generate a comprehensive report"
    
    print("\n🚀 Starting workflow with checkpointing...\n")
    print(f"Input: {user_input}\n")
    
    # 워크플로우 실행 및 체크포인트 저장
    print("Workflow execution with automatic checkpointing:")
    async for event in checkpoint_workflow.run_stream(user_input):
        print(f"Event: {event}\n")
    
    print(f"\n✅ Workflow completed successfully")
    print(f"📁 Checkpoints saved in: {checkpoint_dir}")
    
    # 저장된 체크포인트 확인
    checkpoint_files = list(checkpoint_dir.glob("*.json"))
    if checkpoint_files:
        print(f"\nSaved checkpoint files: {len(checkpoint_files)}")
        for ckpt_file in checkpoint_files:
            print(f"  - {ckpt_file.name}")

await demo_checkpointing()


🚀 Starting workflow with checkpointing...

Input: Analyze cloud computing adoption trends and generate a comprehensive report

Workflow execution with automatic checkpointing:
Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)

Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)

Event: ExecutorInvokedEvent(executor_id=DataCollector, data=Analyze cloud computing adoption trends and generate a comprehensive report)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages=)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages=###)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages= Comprehensive)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages= Report)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages= on)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages= Cloud)

Event: AgentRunUpdateEvent(executor_id=DataCollector, messages= C

## 마무리

### 학습한 내용
1. ✅ WorkflowBuilder를 사용한 워크플로우 구성
2. ✅ 순차적 실행 흐름 (Sequential Flow)
3. ✅ 조건부 라우팅 (Conditional Routing)
4. ✅ 병렬 처리 (Parallel Processing)
5. ✅ 체크포인팅 (Checkpointing)
6. ✅ 상태 관리 (State Management)

### 워크플로우의 장점
- **모듈성**: 재사용 가능한 컴포넌트
- **타입 안정성**: 강력한 타입 체크
- **유연성**: 동적 실행 경로
- **신뢰성**: 체크포인팅 및 복구
- **확장성**: 복잡한 로직 구조화

### 다음 단계
- `04_Research.ipynb`: 실제 웹 검색 통합
- `05_Code_Interpreter.ipynb`: 코드 생성 및 실행